# Zero-shot image-text retrieval evaluation

In [1]:
!python --version

Python 3.10.12


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
%cd /workspace/ViCLIP-OT

/workspace/ViCLIP-OT


## Import libraries

In [4]:
import json
import os
import sys
from collections import defaultdict
from typing import Any, Literal

import numpy as np
import torch
from loguru import logger
from PIL import Image
from pydantic import BaseModel
from torch import Tensor
from torch.utils.data import Dataset

## Config

In [5]:
DATASET_DIR = "./data/UIT-OpenViIC"
DATASET_EMBEDDINGS_DIR = "./data/UIT-OpenViIC-embeddings"
METADATA_JSON_FILE = "test.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load dataset

In [6]:
class ImageTextDataImage(BaseModel):
    id: int | str
    image_path: str


class ImageTextDataAnnotation(BaseModel):
    id: int | str
    caption: str
    image_id: int | str


class ImageTextData(BaseModel):
    images: list[ImageTextDataImage]
    annotations: list[ImageTextDataAnnotation]


class ImageTextDataset(Dataset[tuple[Image.Image | Tensor, list[str], int, int]]):
    """
    Dataset structure:

    ```
    dataset_root/
    ├── images
    │    ├── 000001.jpg
    │    ├── 000002.png
    │    ├── ...
    │    └── nnnnnn.jpg
    ├── train.json
    ├── test.json
    └── val.json
    ```

    Where `train.json`, `test.json`, and `val.json` are metadata files and have the format:
    ```json
    {
        "images": [
            {"id": 1, "image_path": "images/000001.jpg"},
            {"id": 2, "image_path": "images/000002.png"},
            ...
        ],
        "annotations": [
            {"id": 1, "caption": "A caption for image 1", "image_id": 1},
            {"id": 2, "caption": "A caption for image 2", "image_id": 2},
            ...
        ]
    }
    ```
    """

    def __init__(
        self,
        root_dir: str,
        metadata_json_file: str,
        image_transforms=None,
        model_fmt: Literal["gemma", "e5", "qwen3", "bge", "sbert"] = "gemma",
    ) -> None:
        self.root_dir = root_dir
        self.metadata_file_path = os.path.join(self.root_dir, metadata_json_file)
        self.image_transforms = image_transforms
        self.model_fmt = model_fmt

        logger.info(f"Loading image text data from: {self.metadata_file_path}")
        with open(self.metadata_file_path, "r") as f:
            self.metadata = ImageTextData.model_validate(json.load(f))

        logger.info(
            f"Found {len(self.metadata.images)} images and {len(self.metadata.annotations)} annotations."
        )
        self.id_to_image_path = {image.id: image.image_path for image in self.metadata.images}

        captions_by_image_id: dict[int, list[tuple[int, str]]] = defaultdict(list)
        for annotation in self.metadata.annotations:
            image_id = annotation.image_id
            if image_id not in self.id_to_image_path:
                raise RuntimeError(
                    f"Could not find image with ID {image_id} for annotation {annotation.id}"
                )

            captions_by_image_id[image_id].append((annotation.id, annotation.caption))

        # list of list of pair ids for each sample index
        self.pair_ids_by_sample_index: list[list[int]] = []

        # samples: (imag_id, image_path, list of captions)
        self.samples: list[tuple[int, str, list[str]]] = []
        pair_count = 0
        for image_id in sorted(captions_by_image_id.keys()):
            captions_by_image_id[image_id].sort(key=lambda x: x[0])  # sort by caption_id
            captions = [caption for _caption_id, caption in captions_by_image_id[image_id]]
            image_path = os.path.join(self.root_dir, self.id_to_image_path[image_id])
            self.samples.append((image_id, image_path, captions))

            self.pair_ids_by_sample_index.append(
                list(range(pair_count, pair_count + len(captions)))
            )
            pair_count += len(captions)

    def get_pair_ids(self, indices: list[int]) -> list[int]:
        """
        Given a list of sample indices, return the corresponding list of pair ids.

        This is useful for retrieving caption (and image) embeddings
        from some pre-computed embedding matrix.
        """
        p_ids: list[int] = []
        for idx in indices:
            p_ids.extend(self.pair_ids_by_sample_index[idx])

        return p_ids

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx) -> tuple[Image.Image | Tensor, list[str], int, int]:
        image_id, image_path, captions = self.samples[idx]

        try:
            image = Image.open(image_path)
            # handle palette images with transparency
            if image.mode == "P" and "transparency" in image.info:
                image = image.convert("RGBA")

            image = image.convert("RGB")

        except (OSError, SyntaxError) as e:
            logger.warning(f"Corrupt image at {image_path}, skipping. Error: {e}")
            # recursively get the next image
            return self.__getitem__((idx + 1) % len(self))

        if self.image_transforms is not None:
            image = self.image_transforms(image)

        formatted_captions = None
        if self.model_fmt == "gemma":
            # https://huggingface.co/google/embeddinggemma-300m#prompt-instructions
            # TODO: this prompt is for encode document, consider supporting encode for query.
            formatted_captions = [
                f"sentence similarity | query: {caption}" for caption in captions
            ]

        elif self.model_fmt == "e5":
            # https://huggingface.co/intfloat/multilingual-e5-base#usage
            formatted_captions = [f"query: {caption}" for caption in captions]

        elif self.model_fmt == "qwen3":

            def get_detailed_instruct(task_description: str, query: str) -> str:
                return f"Instruct: {task_description}\nQuery:{query}"

            task = "Given a web search query, retrieve relevant passages that answer the query"

            # https://huggingface.co/qwen/qwen3-embedding-0.6b#usage
            formatted_captions = [
                f"{get_detailed_instruct(task, caption)}" for caption in captions
            ]

        elif self.model_fmt == "bge":
            # https://huggingface.co/baai/bge-m3#usage
            # BGE does not require special formatting
            formatted_captions = [f"{caption}" for caption in captions]

        elif self.model_fmt == "sbert":
            # https://www.sbert.net/docs/usage/semantic_textual_similarity.html
            # SBERT does not require special formatting
            formatted_captions = [f"{caption}" for caption in captions]

        else:
            raise ValueError(
                f"Invalid model_fmt: {self.model_fmt}. "
                f"Expected one of ['gemma', 'e5', 'qwen3', 'bge', 'sbert']"
            )

        return image, formatted_captions, int(image_id), idx

In [7]:
DATASET_DIR = DATASET_DIR
metadata_json_file = METADATA_JSON_FILE
metadata_file_path = os.path.join(DATASET_DIR, metadata_json_file)

logger.info(f"Loading image text data from: {metadata_file_path}")
with open(metadata_file_path, "r") as f:
    metadata = ImageTextData.model_validate(json.load(f))

logger.info(f"Found {len(metadata.images)} images and {len(metadata.annotations)} annotations.")

id_to_image_path = {image.id: image.image_path for image in metadata.images}

samples: list[tuple[str, int, int, str]] = []
for annotation in metadata.annotations:
    image_id = annotation.image_id
    if image_id not in id_to_image_path:
        raise RuntimeError(
            f"Could not find image with ID {image_id} for annotation {annotation.id}"
        )

    image_path = os.path.join(DATASET_DIR, id_to_image_path[image_id])
    samples.append((image_path, image_id, annotation.id, annotation.caption))

# sorted by image_id, annotation_id
samples.sort(key=lambda x: (x[1], x[2]))

sorted_image_ids = [image_id for _, image_id, _, _ in samples]

2026-02-05 00:59:02.096 | INFO     | __main__:<module>:5 - Loading image text data from: ./data/UIT-OpenViIC/test.json
2026-02-05 00:59:02.117 | INFO     | __main__:<module>:9 - Found 2001 images and 10001 annotations.


In [8]:
sorted_image_ids[:20]

[245,
 245,
 245,
 245,
 245,
 3989,
 3989,
 3989,
 3989,
 3989,
 5194,
 5194,
 5194,
 5194,
 5194,
 11742,
 11742,
 11742,
 11742,
 11742]

In [9]:
def _compute_retrieval_metrics(
    logits: Tensor, mask: Tensor, prefix: str, k_vals=(1, 5, 10)
) -> dict[str, Any]:
    """ "Compute recall@k and mean rank."""

    results = {}
    max_k = min(max(k_vals), logits.shape[1])
    _, top_indices = logits.topk(max_k, dim=1)  # [B, max_k]

    # gather ground truth booleans at the retrieved positions
    rows = torch.arange(logits.shape[0]).view(-1, 1)
    retrieved_mask = mask[rows, top_indices]  # [B, max_k]

    for k in k_vals:
        # hit if at least one of the top k is True
        hits = retrieved_mask[:, :k].any(dim=1)
        results[f"{prefix}_R__{k}"] = hits.float().mean().item()

    argsort = torch.argsort(logits, dim=1, descending=True)

    # sorted_mask[i, j] is True if the item at rank 'j' is a match
    sorted_mask = mask[rows, argsort]

    # find the first rank (min index) where sorted_mask is True
    rank_matrix = torch.arange(logits.shape[1]).view(1, -1).float()
    masked_ranks = rank_matrix.expand(logits.shape[0], -1).clone()
    masked_ranks[~sorted_mask] = float("inf")

    # get the "Best Rank" (lowest index) for every row
    best_rank_per_row = masked_ranks.min(dim=1).values

    # convert 0-indexed to 1-indexed
    best_rank_per_row = best_rank_per_row.numpy() + 1

    results[f"{prefix}_mean_rank"] = np.mean(best_rank_per_row)
    results[f"{prefix}_median_rank"] = np.floor(np.median(best_rank_per_row))

    return results


def get_retrieval_metrics(
    image_features: Tensor,
    text_features: Tensor,
    logit_scale: Tensor,
    image_ids: Tensor | None = None,
) -> dict[str, Any]:
    """
    If `image_ids` is provided, the computation will take into account image with multiple captions.
    """
    metrics: dict[str, Any] = {}
    if image_ids is None:
        # 1-1 image caption mapping
        image_ids = torch.arange(len(image_features))

    image_features = image_features.cpu().float()
    text_features = text_features.cpu().float()
    image_ids = image_ids.cpu()
    image_ids = image_ids.cpu()

    unique_ids, first_indices = np.unique(image_ids.numpy(), return_index=True)
    unique_ids = torch.from_numpy(unique_ids)
    first_indices = torch.from_numpy(first_indices)

    unique_image_features = image_features[first_indices]

    # Image-to-Text
    # Query:   Unique Images [N_unique]
    # Gallery: All Texts     [N_total]
    # Protocol: For each unique image, did we find ANY of its captions?
    # Logits: [N_unique, N_total]
    logits_i2t = logit_scale * unique_image_features @ text_features.t()

    # Mask: [N_unique, N_total]
    # Rows are Unique IDs, Cols are All IDs. Match if they are equal.
    mask_i2t = unique_ids.view(-1, 1) == image_ids.view(1, -1)

    metrics.update(_compute_retrieval_metrics(logits_i2t, mask_i2t, prefix="i2t"))

    # Text-to-Image
    # Query:   All Texts     [N_total]
    # Gallery: Unique Images [N_unique]
    # Protocol: For each caption, did we find the ONE correct image?
    # Logits: [N_total, N_unique]
    logits_t2i = logit_scale * text_features @ unique_image_features.t()

    # Mask: [N_total, N_unique]
    # Rows are All IDs, Cols are Unique IDs. Match if they are equal.
    mask_t2i = image_ids.view(-1, 1) == unique_ids.view(1, -1)

    metrics.update(_compute_retrieval_metrics(logits_t2i, mask_t2i, prefix="t2i"))

    return metrics

## Qwen3-VL-Embedding-2B

In [10]:
caption_embeddings = torch.load(os.path.join(DATASET_EMBEDDINGS_DIR, 'test_caption_embeddings_qwen3_vl_embedding_2b.pt'), map_location=device)
image_embeddings = torch.load(os.path.join(DATASET_EMBEDDINGS_DIR, 'test_image_embeddings_qwen3_vl_embedding_2b.pt'), map_location=device)
assert caption_embeddings.shape[0] == len(samples)
caption_embeddings.shape, image_embeddings.shape

(torch.Size([10001, 2048]), torch.Size([10001, 2048]))

In [11]:
metrics = get_retrieval_metrics(
    image_embeddings,
    caption_embeddings,
    logit_scale=torch.tensor(1.0),
    image_ids=torch.tensor(sorted_image_ids),
)

In [12]:
metrics

{'i2t_R__1': 0.3983008563518524,
 'i2t_R__5': 0.6651673913002014,
 'i2t_R__10': 0.7701149582862854,
 'i2t_mean_rank': np.float32(15.169915),
 'i2t_median_rank': np.float32(2.0),
 't2i_R__1': 0.32126787304878235,
 't2i_R__5': 0.540045976638794,
 't2i_R__10': 0.6293370723724365,
 't2i_mean_rank': np.float32(49.519848),
 't2i_median_rank': np.float32(4.0)}

## Qwen3-VL-Embedding-8B

In [13]:
caption_embeddings = torch.load(os.path.join(DATASET_EMBEDDINGS_DIR, 'test_caption_embeddings_qwen3_vl_embedding_8b.pt'), map_location=device)
image_embeddings = torch.load(os.path.join(DATASET_EMBEDDINGS_DIR, 'test_image_embeddings_qwen3_vl_embedding_8b.pt'), map_location=device)
assert caption_embeddings.shape[0] == len(samples)
caption_embeddings.shape, image_embeddings.shape

(torch.Size([10001, 4096]), torch.Size([10001, 4096]))

In [14]:
metrics = get_retrieval_metrics(
    image_embeddings,
    caption_embeddings,
    logit_scale=torch.tensor(1.0),
    image_ids=torch.tensor(sorted_image_ids),
)

In [15]:
metrics

{'i2t_R__1': 0.5997001528739929,
 'i2t_R__5': 0.8345826864242554,
 'i2t_R__10': 0.9030484557151794,
 'i2t_mean_rank': np.float32(5.743628),
 'i2t_median_rank': np.float32(1.0),
 't2i_R__1': 0.4378562271595001,
 't2i_R__5': 0.6640335917472839,
 't2i_R__10': 0.7483251690864563,
 't2i_mean_rank': np.float32(28.067493),
 't2i_median_rank': np.float32(2.0)}

## Jina-Embedding-v4

In [16]:
caption_embeddings = torch.load(os.path.join(DATASET_EMBEDDINGS_DIR, 'test_caption_embeddings_jina_embeddings_v4.pt'), map_location=device)
image_embeddings = torch.load(os.path.join(DATASET_EMBEDDINGS_DIR, 'test_image_embeddings_jina_embeddings_v4.pt'), map_location=device)
assert caption_embeddings.shape[0] == len(samples)
caption_embeddings.shape, image_embeddings.shape

(torch.Size([10001, 2048]), torch.Size([10001, 2048]))

In [17]:
metrics = get_retrieval_metrics(
    image_embeddings,
    caption_embeddings,
    logit_scale=torch.tensor(1.0),
    image_ids=torch.tensor(sorted_image_ids),
)

In [18]:
metrics

{'i2t_R__1': 0.41479259729385376,
 'i2t_R__5': 0.6676661372184753,
 'i2t_R__10': 0.7561219334602356,
 'i2t_mean_rank': np.float32(18.994503),
 'i2t_median_rank': np.float32(2.0),
 't2i_R__1': 0.23967602849006653,
 't2i_R__5': 0.42215779423713684,
 't2i_R__10': 0.5029497146606445,
 't2i_mean_rank': np.float32(100.41606),
 't2i_median_rank': np.float32(10.0)}

## Jina CLIP v2

In [19]:
caption_embeddings = torch.load(os.path.join(DATASET_EMBEDDINGS_DIR, 'test_caption_embeddings_jina_clip_v2.pt'), map_location=device)
image_embeddings = torch.load(os.path.join(DATASET_EMBEDDINGS_DIR, 'test_image_embeddings_jina_clip_v2.pt'), map_location=device)
assert caption_embeddings.shape[0] == len(samples)
caption_embeddings.shape, image_embeddings.shape

(torch.Size([10001, 1024]), torch.Size([10001, 1024]))

In [20]:
metrics = get_retrieval_metrics(
    image_embeddings,
    caption_embeddings,
    logit_scale=torch.tensor(1.0),
    image_ids=torch.tensor(sorted_image_ids),
)

In [21]:
metrics

{'i2t_R__1': 0.40229883790016174,
 'i2t_R__5': 0.6501749157905579,
 'i2t_R__10': 0.7441279292106628,
 'i2t_mean_rank': np.float32(19.284357),
 'i2t_median_rank': np.float32(2.0),
 't2i_R__1': 0.30006998777389526,
 't2i_R__5': 0.5209479331970215,
 't2i_R__10': 0.6170383095741272,
 't2i_mean_rank': np.float32(50.052197),
 't2i_median_rank': np.float32(5.0)}